# RealityCheck GenImage v2 — validation-only SID false-positive calibration

This notebook does **not retrain** the model. It selects one decision threshold using clean validation predictions only, minimizing SID false positives while requiring at least 95% SID generated-image recall and preserving GenImage performance. It then exploratorily re-scores the locked threshold on the already-saved test predictions. Those tests were previously inspected at threshold 0.50, so they are not presented as a fresh unbiased holdout.

Before **Run All**: first make sure the calibration code is committed and pushed. Start a fresh Kaggle session, enable a **T4 GPU** and **Internet**, attach `cartografia/unbiased-tiny-genimage` version 1, and attach a private Kaggle input containing your original file named exactly `genimage_v2_export.zip`. WildFake is never read.

While it runs, **Cancel Run** and an active cell mean it is still working; the Output GiB number is disk usage, not progress. Success is the line `Validated calibration export ready`. A fresh run normally takes roughly 20–45 minutes.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import torch

REPO_URL = "https://github.com/LINGSIHAN/TikTok-Hackathon-Track-5.git"
BRANCH = "master"
PROJECT_DIR = Path("/kaggle/working/TikTok-Hackathon-Track-5-calibration")
DATASET_ROOT = Path("/kaggle/input/unbiased-tiny-genimage")
LICENSE_CONFIRMED = True

def run(*args):
    command = [str(value) for value in args]
    print("+", " ".join(command), flush=True)
    subprocess.run(command, check=True)

if not LICENSE_CONFIRMED:
    raise RuntimeError("Dataset permission must be confirmed before calibration.")
if not DATASET_ROOT.is_dir():
    raise RuntimeError("Use Add Input and attach cartografia/unbiased-tiny-genimage version 1.")
if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0).upper():
    raise RuntimeError("Choose a T4 GPU in Kaggle Settings, restart, and run all cells again.")
print("GPU:", torch.cuda.get_device_name(0))

preferred = Path("/kaggle/working/genimage_v2_export.zip")
zip_candidates = [preferred] if preferred.is_file() else sorted(Path("/kaggle/input").rglob("genimage_v2_export.zip"))
if zip_candidates:
    if len(zip_candidates) != 1:
        raise RuntimeError(f"Found multiple candidate ZIPs: {zip_candidates}")
    CANDIDATE_INPUT = zip_candidates[0]
else:
    search_roots = [Path("/kaggle/working"), Path("/kaggle/input")]
    checkpoints = []
    for root in search_roots:
        checkpoints.extend(root.rglob("model_v2.safetensors"))
    extracted_roots = sorted({path.parents[2] for path in checkpoints if path.parent.name == "checkpoints" and path.parent.parent.name == "local_audit"})
    if len(extracted_roots) != 1:
        raise RuntimeError("Attach a private Kaggle input containing the original genimage_v2_export.zip, then Run All again.")
    CANDIDATE_INPUT = extracted_roots[0]
print("Candidate input:", CANDIDATE_INPUT)

In [ ]:
if PROJECT_DIR.exists():
    if not (PROJECT_DIR / ".git").is_dir():
        raise RuntimeError(f"{PROJECT_DIR} exists but is not the expected Git checkout.")
    origin = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "remote", "get-url", "origin"], text=True).strip()
    if origin.rstrip("/").removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError(f"Unexpected existing repository origin: {origin}")
    tracked = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"], text=True).strip()
    if tracked:
        raise RuntimeError("Tracked changes exist in the Kaggle checkout; use a fresh session.")
    run("git", "-C", PROJECT_DIR, "fetch", "--depth", "50", "origin", BRANCH)
    run("git", "-C", PROJECT_DIR, "merge", "--ff-only", "FETCH_HEAD")
else:
    run("git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, PROJECT_DIR)

os.chdir(PROJECT_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository commit:", COMMIT)
run(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt")

In [ ]:
run(
    sys.executable,
    "scripts/run_genimage_v2_calibration_kaggle.py",
    "--input-root", DATASET_ROOT,
    "--candidate-input", CANDIDATE_INPUT,
    "--output-root", "/kaggle/working",
    "--license-confirmed",
)

In [ ]:
archive = Path("/kaggle/working/genimage_v2_calibration_export.zip")
if not archive.is_file() or archive.stat().st_size == 0:
    raise RuntimeError("The validated calibration export was not created. Review the first failed cell.")
print("Download this file from Kaggle Output:", archive)